# 5G Network Intelligence - Nested Historical Exceedance Probability

## Hypothesis and prior evidence

Independent P90/P95 classifiers have useful probability signal but incoherent nested-event pairs and weak fixed-0.5 F1. Reuse the pinned P90 classifier; train only P(P95 exceedance | P90 exceedance, history). Multiply the two probabilities. This guarantees nesting by construction, but may worsen Brier, log loss or ranking. No improvement is assumed.

## Temporal protocol

Keep the two original fit/stop/score windows, ending by hour 503. Reuse saved fit-prefix thresholds unchanged. Observed warning labels filter conditional fit/stopping samples only; they never enter features or inference-time routing. Scoring includes all eligible targets, including non-warning events. Reused windows are not fresh-test evidence. Validation 504-671, final holdout and production remain excluded.


In [ ]:
RUN_TRAINING = False
SOURCE_COMMIT = 'a853d4af7f6223083729dfcff9e38df56390ec8f'


## Fixed model and metrics

Six additional binary LightGBM fits: two folds and three horizons, with unchanged 40 causal features and original hyperparameters. No class weighting, threshold search or calibration. Compare nested P95 probabilities with the saved independent P95 model on identical targets. Report Brier, log loss, average precision and fixed-0.5 classification metrics, including regressions. Warning probabilities are unchanged. This study does not improve or replace the original MAE/WAPE objective.


In [ ]:
PROTOCOL = {'protocol': 'nested-historical-exceedance-probability-v1',
 'source_manifest_sha256': '561eeb3f9c8c438cf752ef52dfd2e14e77bf31b1659c140e2bf16cdaabb47db6',
 'base_protocol': 'configs/risk_probability.yaml',
 'warning_classifier': 'reuse_unchanged',
 'critical_classifier': 'conditional_on_observed_warning_labels_in_fit_and_stopping_only',
 'combination': 'probability_product_chain_rule',
 'threshold_refitting': 'forbidden',
 'class_reweighting': 'forbidden',
 'probability_calibration': 'none',
 'decision_threshold': 0.5,
 'threshold_selection': 'forbidden',
 'validation_504_671': 'forbidden',
 'original_final_holdout': 'forbidden',
 'production_change': 'forbidden',
 'claim_scope': 'one_seed_reused_development_probability_study_not_MAE_improvement'}


## Environment and source

Attach the completed original probability worker for canonical data, pinned probability artifacts and inherited causal feature code. A separate private MIT source dataset supplies the new trainer, probability metrics and pinned configurations. Full source remains in named modules.


In [ ]:
from pathlib import Path
import os,sys,shutil,subprocess,tarfile
if RUN_TRAINING:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Hosted Kaggle required')
    roots=[p.parent.parent for p in Path('/kaggle/input').rglob('risk_probability_development.csv') if (p.parent.parent/'data/processed/panel.npz').exists()]
    if len(roots)!=1:
        raise RuntimeError('Attach exactly one completed canonical source')
    SOURCE=roots[0]/'artifacts/risk_probability'
    PROJECT=Path('/kaggle/working/nested_risk_probability')
    PROJECT.mkdir(exist_ok=True)
    for folder in ('src','scripts','configs','data/processed'):
        shutil.copytree(roots[0]/folder,PROJECT/folder,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
    os.chdir(PROJECT)
    sys.path.insert(0,str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT']=SOURCE_COMMIT
    patches=list(Path('/kaggle/input').rglob('train_nested_risk_probability.py'))
    if len(patches)!=1:
        raise RuntimeError('Attach exactly one probability study source')
    patch=patches[0].parent
    for name,folder in (('train_nested_risk_probability.py','scripts'),('nested_probability.py','src/risk'),('probability_metrics.py','src/risk'),('nested_risk_probability.yaml','configs'),('risk.yaml','configs')):
        shutil.copyfile(patch/name,Path(folder)/name)
    Path('reports').mkdir(exist_ok=True)


## Hosted training and debugging

Reuse fold thresholds unchanged; save conditional checkpoints, stopping rounds, score arrays and provenance. Preserve missing targets as missing labels; do not score only warning-positive targets. Save outputs after every classifier; a failure must retain logs rather than report success.


In [ ]:
if RUN_TRAINING:
    with Path('reports/nested_risk_probability_training.log').open('w') as log:
        process=subprocess.Popen([sys.executable,'-u','-m','scripts.train_nested_risk_probability','--source-root',str(SOURCE)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line)
            log.flush()
        status=process.wait()
    with tarfile.open('/kaggle/working/nested_risk_probability_results.tar.gz','w:gz') as archive:
        for folder in ('reports','artifacts/nested_risk_probability'):
            if Path(folder).exists():
                archive.add(folder,arcname=folder)
    if status:
        raise RuntimeError(f'Hosted probability study failed: {status}; inspect saved logs')
else:
    print('Review only: no fitting or calibration.')


## Results and limitations

Report both thresholds at every horizon/fold, eligible counts and undefined metrics. Better ranking or Brier score does not imply calibrated deployment probabilities or a useful operating point. Historical quantiles are not operator limits or congestion labels. A later unseen-data study is required before any production change.


In [ ]:
paths=[Path('reports/nested_risk_probability_development.csv'),Path('../reports/nested_risk_probability_development.csv')]
if Path('/kaggle/input').is_dir():
    paths+=list(Path('/kaggle/input').rglob('nested_risk_probability_development.csv'))
path=next((p for p in paths if p.is_file()),None)
if path:
    import pandas as pd
    from IPython.display import display
    display(pd.read_csv(path))
else:
    print('No completed probability report attached.')
